# 🔥 Mojo/MAX M2 prototypes on Colab T4 (issue #57)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/mojo_max_m2_t4.ipynb)

Runs the three M2 correctness/executability prototypes from [`docs/research/mojo-max/m1-responsibility-map.md`](../docs/research/mojo-max/m1-responsibility-map.md) on a real Colab T4, unchanged from the versions already run on this project's M1 (Apple Silicon):

1. **Snake1d** ( + (\alpha+\epsilon)^{-1}\sin(\alpha x)^2$) — PASSED on M1, confirms the FP16-overflow hypothesis directly.
2. **Conv1d via `ops.conv2d`** (route A, degenerate height axis) — PASSED on M1, no custom Mojo kernel needed.
3. **ConvTranspose1d** — CPU PASSED on M1, but **GPU (Metal) hard-crashed** with `symbol not found: cudnnCreate` (attempting to load NVIDIA's cuDNN on Apple Silicon). T4 has genuine CUDA/cuDNN — **this notebook is the only way to find out whether that crash is Metal-specific or a general MAX limitation.**

Minimal blocker: only `pixi`+`modular` gets installed here, matching the M0 T4 notebook's convention — no TTS/STT/Qwen stack.


## 1. GPU and driver


In [ ]:
!nvidia-smi


In [ ]:
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name,compute_cap",
                       "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr)


## 2. Install pixi + Mojo/MAX (stable channel 26.5)


In [ ]:
!curl -fsSL https://pixi.sh/install.sh | bash
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.pixi/bin')}:{os.environ['PATH']}"
!pixi --version


In [ ]:
!mkdir -p /content/mojo-probe-t4
!cd /content/mojo-probe-t4 && pixi init . -c https://conda.modular.com/max/ -c conda-forge
!cd /content/mojo-probe-t4 && pixi add modular
!cd /content/mojo-probe-t4 && pixi run python -c "import max; print('max import ok')"


## 3. Fetch the prototype scripts from the repo


In [ ]:
REPO_URL = "https://github.com/vedmalex/higgs-local-test.git"
REPO_REF = "main"

!rm -rf /content/higgs-local-test
!git clone --quiet --depth 1 --branch "" "" /content/higgs-local-test
!ls /content/higgs-local-test/docs/research/mojo-max/*.py


## 4. Run prototype #1: Snake1d


In [ ]:
!cd /content/mojo-probe-t4 && pixi run python /content/higgs-local-test/docs/research/mojo-max/m2_snake1d_prototype.py 2>&1 | tee /content/m2-snake1d-output-t4.txt


## 5. Run prototype #2: Conv1d via ops.conv2d


In [ ]:
!cd /content/mojo-probe-t4 && pixi run python /content/higgs-local-test/docs/research/mojo-max/m2_conv1d_prototype.py 2>&1 | tee /content/m2-conv1d-output-t4.txt


## 6. Run prototype #3: ConvTranspose1d — the critical one

On M1/Metal this crashed the whole process on GPU with `symbol not found: cudnnCreate`. Each case runs as its own subprocess below so one crash (if it happens here too) does not hide the other four results — exactly like the isolation this needed on M1.


In [ ]:
CASES = [
    (16, 16, 8, 0),
    (16, 16, 5, 1),
    (16, 16, 4, 0),
    (16, 16, 2, 0),
    (16, 16, 3, 1),
]

runner = """
import sys
sys.path.insert(0, "/content/higgs-local-test/docs/research/mojo-max")
import numpy as np
from max.driver import CPU, Accelerator
m2 = __import__("m2_convtranspose1d_prototype")
m2.batch = 1
m2.seq_len = 16

idx = int(sys.argv[1])
device_kind = sys.argv[2]
cases = %r
c_in, c_out, stride, output_padding = cases[idx]
kernel = 2 * stride
rng = np.random.default_rng(9012)
x_np = rng.uniform(-1.0, 1.0, size=(1, c_in, 16)).astype(np.float32)
weight_pt = rng.normal(0, 0.1, size=(c_in, c_out, kernel)).astype(np.float32)
filter_rscf = np.transpose(weight_pt, (2, 1, 0))[np.newaxis, ...].copy()

device_obj = CPU() if device_kind == "cpu" else Accelerator()
out = m2.run_on(device_obj, c_in, c_out, kernel, stride, output_padding, x_np, filter_rscf)
print(f"case={idx} device={device_kind} stride={stride} output_padding={output_padding} -> "
      f"shape={out.shape} nan_inf={int(np.sum(~np.isfinite(out)))}")
""" % CASES

with open("/content/run_one_case_t4.py", "w") as f:
    f.write(runner)
print("wrote /content/run_one_case_t4.py")


In [ ]:
import subprocess

results = []
for i in range(5):
    for dev in ("cpu", "gpu"):
        proc = subprocess.run(
            ["pixi", "run", "python", "-u", "/content/run_one_case_t4.py", str(i), dev],
            cwd="/content/mojo-probe-t4", capture_output=True, text=True,
        )
        line = f"--- case {i} {dev} (exit={proc.returncode}) ---\n{proc.stdout}{proc.stderr}"
        print(line)
        results.append(line)

with open("/content/m2-convtranspose1d-output-t4.txt", "w") as f:
    f.write("\n".join(results))
print("Saved /content/m2-convtranspose1d-output-t4.txt")


## 7. Download results

Download the three `.txt` files below, save them as `docs/research/mojo-max/m2-*-output-t4.txt` locally, and update the corresponding `m2-*-results.md` + issue #57 with the real T4 numbers. Do not embellish: if ConvTranspose1d GPU also crashes on T4, that is the honest, valuable result — it means the whole upsample stage needs a CPU-bound or custom-kernel plan on both targets, not just Apple Silicon.


In [ ]:
from google.colab import files
files.download("/content/m2-snake1d-output-t4.txt")
files.download("/content/m2-conv1d-output-t4.txt")
files.download("/content/m2-convtranspose1d-output-t4.txt")
